In [ ]:
!pip install streamlit pyngrok transformers torch accelerate sentencepiece


In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("")  # 👈 paste your token here


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%%writefile app.py


import streamlit as st
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import requests
import os
from dotenv import load_dotenv

load_dotenv('/content/drive/MyDrive/P1/.env')
GROQ_<REDACTED_SECRET>("")


st.set_page_config(page_title="Customer Feedback Dashboard", layout="wide")
st.title("💬 Customer Feedback Dashboard")
st.markdown("Analyze *Customer Satisfaction (CSAT)*, *Effort (CES)*, *Sentiment*, and generate **empathetic AI replies** ")

with st.sidebar:
    st.header("⚙️ Dashboard Settings")
    st.markdown("Configure model, API, and preferences below 👇")

    api_choice = st.radio(
        "🌐 Choose AI Provider",
        ["Mistral (Recommended)", "Groq (Fallback)"],
        index=0,
        help="Select which AI model to use for generating replies."
    )

    if api_choice == "Mistral (Recommended)":
        MISTRAL_<REDACTED_SECRET>(
            "Mistral API Key",
            type="password",
            placeholder="mistral_xxxxxxxxxxxxxxxxxxx"
        )

        model_choice = st.selectbox(
            "Mistral Model",
            ["mistral-small-latest", "mistral-medium-latest", "mistral-large-latest"],
            index=1
        )
        st.caption("💡 *Small* = fast, *Medium* = balanced, *Large* = best quality")
    else:
        MISTRAL_API_KEY = ""
        model_choice = None

    st.markdown("---")
    st.sidebar.info("💬 You can manually switch between **Mistral API** and **Groq (LLaMA 3.1 8B-Instant)** ")



@st.cache_resource(show_spinner=False)
def load_main_models():
    """Load CSAT, CES, and sentiment models (cached)."""
    csat_model_path = ""
    ces_model_path  = ""

    csat_tokenizer = AutoTokenizer.from_pretrained(csat_model_path)
    csat_model = AutoModelForSequenceClassification.from_pretrained(csat_model_path)

    ces_tokenizer = AutoTokenizer.from_pretrained(ces_model_path)
    ces_model = AutoModelForSequenceClassification.from_pretrained(ces_model_path)

    sentiment_analyzer = pipeline(
        "sentiment-analysis",
        model="nlptown/bert-base-multilingual-uncased-sentiment"
    )

    return csat_tokenizer, csat_model, ces_tokenizer, ces_model, sentiment_analyzer


csat_tokenizer, csat_model, ces_tokenizer, ces_model, sentiment_analyzer = load_main_models()

# Lazy load translator
translator = None
def get_translator():
    """Load translation model only when needed."""
    global translator
    if translator is None:
        # with st.spinner("🌐 Loading translation model..."):
            translator = pipeline(
                "translation",
                model="Helsinki-NLP/opus-mt-en-ROMANCE",
                device="cpu"
            )
    return translator


def predict_csat(text):
    inputs = csat_tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = csat_model(**inputs)
    pred = torch.argmax(outputs.logits, dim=1).item()
    return ["Dissatisfied", "Neutral", "Satisfied"][pred]


def predict_ces(text):
    inputs = ces_tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = ces_model(**inputs)
    pred = torch.argmax(outputs.logits, dim=1).item()
    return ["Difficult", "Neutral", "Easy"][pred]


def analyze_sentiment(text):
    result = sentiment_analyzer(text)[0]
    return result["label"], result["score"]

MISTRAL_API_URL = ""
GROQ_API_URL = "s"

def generate_ai_reply(feedback, csat, ces, sentiment):
    """Generate empathetic replies using Mistral API, fallback to Groq if needed."""
    sentiment_lower = sentiment.lower()

    # Tone detection
    if any(x in sentiment_lower for x in ["1 star", "2 star", "negative", "dissatisfied"]):
        tone = "apologetic and helpful"
        mood_instruction = "Acknowledge the issue, apologize sincerely, and offer help or compensation."
    elif "3 star" in sentiment_lower or "neutral" in sentiment_lower:
        tone = "understanding and professional"
        mood_instruction = "Thank them for the feedback and mention that improvements will be made."
    else:
        tone = "warm and appreciative"
        mood_instruction = "Thank them genuinely and express happiness for their satisfaction."

    prompt = f"""
    You are a {tone} customer support assistant.
    Customer Feedback: "{feedback}"
    Predicted CSAT: {csat}
    Predicted CES: {ces}
    Sentiment: {sentiment}

    Task: {mood_instruction}
    Write only one short, natural, empathetic customer support reply (1–2 sentences max).
    """

    # --- 1️⃣ Try Mistral API first (if key provided)
    if MISTRAL_API_KEY:
        try:
            headers = {"Authorization": f"Bearer {MISTRAL_API_KEY}", "Content-Type": "application/json"}
            data = {
                "model": model_choice,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": 150,
                "temperature": 0.7,
                "top_p": 0.9
            }
            response = requests.post(MISTRAL_API_URL, headers=headers, json=data, timeout=25)
            response.raise_for_status()
            return response.json()["choices"][0]["message"]["content"].strip()
        except Exception as e:
            print(f"⚠️ Mistral failed: {e} → switching to Groq fallback...")

   

    try:
        headers = {"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"}
        data = {
            "model": "llama-3.1-8b-instant",  # ✅ Latest Groq model (fast and supported)
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 150,
            "temperature": 0.7
        }
        response = requests.post("", headers=headers, json=data, timeout=25)
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"].strip()
    except Exception as e:
        return f"⚠️ Both APIs failed: {e}"



def full_prediction_pipeline(english_text):
    translator = get_translator()
    translated = translator(english_text)[0]["translation_text"]

    csat_result = predict_csat(translated)
    ces_result = predict_ces(translated)
    sentiment_label, sentiment_score = analyze_sentiment(english_text)
    ai_reply = generate_ai_reply(english_text, csat_result, ces_result, sentiment_label)

    return {
        "original_text": english_text,
        "translated_text": translated,
        "csat": csat_result,
        "ces": ces_result,
        "sentiment": sentiment_label,
        "sentiment_score": sentiment_score,
        "ai_reply": ai_reply
    }


user_text = st.text_area(
    "📝 Enter customer feedback:",
    height=150,
    placeholder="Type or paste feedback here..."
)

if st.button("🔍 Analyze Feedback", use_container_width=True):
    if user_text.strip():
        with st.spinner("Analyzing feedback... Please wait..."):
            result = full_prediction_pipeline(user_text)
        st.markdown("<div style='text-align:center; font-size:20px; color:#00C853;'>✅ Analysis Complete!</div>", unsafe_allow_html=True)
        st.markdown("---")
        st.subheader("📊 Analysis Results")

        st.markdown(f"🗣 **Original Text:** {result['original_text']}")
        # st.markdown(f"🌍 **Translated (Portuguese):** {result['translated_text']}")

        # Metrics
        col1, col2, col3 = st.columns(3)
        csat_color = "green" if result["csat"] == "Satisfied" else "orange" if result["csat"] == "Neutral" else "red"
        ces_color = "green" if result["ces"] == "Easy" else "orange" if result["ces"] == "Neutral" else "red"

        col1.markdown(f"### 💡 CSAT (Satisfaction)\n<span style='color:{csat_color};font-size:24px'>{result['csat']}</span>", unsafe_allow_html=True)
        col2.markdown(f"### ⚙️ CES (Effort)\n<span style='color:{ces_color};font-size:24px'>{result['ces']}</span>", unsafe_allow_html=True)

        sentiment_map = {
            "1 star": "😡 Very Negative",
            "2 stars": "😞 Negative",
            "3 stars": "😐 Neutral",
            "4 stars": "🙂 Positive",
            "5 stars": "🤩 Very Positive"
        }
        sentiment_display = sentiment_map.get(result["sentiment"], result["sentiment"])
        col3.markdown(f"### 🧠 Sentiment\n{sentiment_display} ({result['sentiment_score']:.2f})")

        # AI reply
        st.markdown("---")
        st.subheader("💬 AI-Suggested Reply")
        st.info(result["ai_reply"])
        if MISTRAL_<REDACTED_SECRET>("⚙️ Reply generated using **Mistral API**")
        else:
            st.caption("⚙️ Reply generated using **Groq Fallback (llama-3.1-8b-instant)**")
   
   
        import pandas as pd
        from io import StringIO
        import os

        if "feedback_records" not in st.session_state:
            st.session_state.feedback_records = []

        st.session_state.feedback_records.append({
            "Feedback": result["original_text"],
            "CSAT": result["csat"],
            "CES": result["ces"],
            "Sentiment": result["sentiment"],
            "Sentiment Score": result["sentiment_score"],
            "AI Reply": result["ai_reply"]
        })

        df_export = pd.DataFrame(st.session_state.feedback_records)

        csv_buffer = StringIO()
        df_export.to_csv(csv_buffer, index=False)
        st.markdown("<div style='text-align:center; margin-top: 30px;'>", unsafe_allow_html=True)
        st.download_button(
            label="📥 Download Feedback Analysis CSV",
            data=csv_buffer.getvalue(),
            file_name="customer_feedback_analysis.csv",
            mime="text/csv",
        )
        st.markdown("</div>", unsafe_allow_html=True)
        save_path = '/content/drive/MyDrive/P1/feedback_history.csv'
        if not os.path.exists(save_path):
            df_export.to_csv(save_path, index=False)
        else:
            df_export.tail(1).to_csv(save_path, mode='a', header=False, index=False)


    else:
        st.warning("⚠️ Please enter some feedback before analyzing.")

st.markdown("---")
st.markdown("<div style='text-align:center;'>", unsafe_allow_html=True)
st.markdown(
    """
    <div style='text-align: center; margin-top: 40px; color: grey;'>
        <p>Powered by 🤗 <b>Transformers</b> | 🚀 <b>Streamlit</b> | 💡 <b>Mistral API</b></p>
    </div>
    """,
    unsafe_allow_html=True
)




In [ ]:

public_url = ngrok.connect(8501)
print("✅ Your Streamlit app is live at:", public_url)
!streamlit run app.py --server.port 8501 &>/dev/null &


In [ ]:
import requests

headers = {
    "Authorization": "<REDACTED_SECRET>",
    "Content-Type": "application/json"
}

data = {
    "model": "llama-3.1-8b-instant",  # ✅ updated model name
    "messages": [
        {"role": "user", "content": "Say hello!"}
    ]
}

r = requests.post("https://api.groq.com/openai/v1/chat/completions", headers=headers, json=data)
print(r.status_code)
print(r.json())


In [ ]:
# 👇 Replace the path below with your project folder if different
env_path = '/content/drive/MyDrive/P1/.env'

# 👇 Write your API keys securely (edit inside the quotes)
with open(env_path, 'w') as f:
    f.write('GROQ_<REDACTED_SECRET>\n')

print(f".env file created at: {env_path}")


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv('/content/drive/MyDrive/P1/.env')
print(os.getenv("GROQ_API_KEY"))


In [ ]:
!pip -q install huggingface_hub

from huggingface_hub import login
login(<REDACTED_SECRET>)   # <-- paste your token


In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path="/content/drive/MyDrive/P1/csat_weighted_model",
    repo_id="",
    repo_type="model"
)

api.upload_folder(
    folder_path="/content/drive/MyDrive/P1/CES_weighted_model_",
    repo_id="",
    repo_type="model"
)
